### ЗАДАЧА: Типизированный каталог сотрудников

HR-команде нужен маленький сервис-каталог сотрудников.
Из него должно быть удобно получить:
- список активных сотрудников,
- сотрудников конкретного отдела,
- email по `emp_id`,
- группировку людей по отделам.

In [ ]:
from dataclasses import dataclass
from typing import Optional, Protocol, Sequence, TypedDict

# rows: emp_id, name, department, email, is_active
rows = [
    {'emp_id': 1, 'name': 'Алиса', 'department': 'Data', 'email': 'alice@company.com', 'is_active': True},
    {'emp_id': 2, 'name': 'Боб', 'department': 'Backend', 'email': 'bob@company.com', 'is_active': True},
    {'emp_id': 3, 'name': 'Кира', 'department': 'Data', 'email': 'kira@company.com', 'is_active': False},
    {'emp_id': 4, 'name': 'Лев', 'department': 'QA', 'email': 'lev@company.com', 'is_active': True},
]

class EmployeeRow(TypedDict):
    emp_id: int
    name: str
    department: str
    email: str
    is_active: bool


class EmployeeRepository(Protocol):
    def get_all(self) -> Sequence[EmployeeRow]:
        pass

    def get_by_id(self, emp_id: int) -> Optional[EmployeeRow]:
        pass

DepartmentMap = dict[str, list[str]]

def build_index(rows: Sequence[EmployeeRow]) -> dict[int, EmployeeRow]:
    index = {}
    for row in rows:
        index[row['emp_id']] = row
    return index

class InMemoryEmployeeRepository:
    def __init__(self, rows: Sequence[EmployeeRow]):
        self._rows = rows

    def get_all(self) -> Sequence[EmployeeRow]:
        return self._rows

    def get_by_id(self, emp_id: int) -> Optional[EmployeeRow]:
        for row in self._rows:
            if row['emp_id'] == emp_id:
                return row
        return None

@dataclass
class EmployeeService:
    repo: EmployeeRepository

    def active_names(self) -> list[str]:
        active_employees = [
            emp['name'] for emp in self.repo.get_all() if emp['is_active']
        ]
        return active_employees

    def department_members(self, department: str) -> list[EmployeeRow]:
        members = [
            emp for emp in self.repo.get_all()
            if emp['department'] == department
        ]
        return members

    def find_email(self, emp_id: int) -> Optional[str]:
        row = self.repo.get_by_id(emp_id)
        if row is None:
            return None
        return row['email']

    def group_by_department(self) -> DepartmentMap:
        department_groups: DepartmentMap = {}
        for emp in self.repo.get_all():
            dept = emp['department']
            if dept not in department_groups:
                department_groups[dept] = []
            department_groups[dept].append(emp['name'])
        return department_groups

repo = InMemoryEmployeeRepository(rows)
service = EmployeeService(repo)

print("=== РЕЗУЛЬТАТЫ РАБОТЫ СЕРВИСА КАТАЛОГА СОТРУДНИКОВ ===\n")

print("1. Индекс по emp_id (build_index):")
index = build_index(rows)
for emp_id, emp_data in index.items():
    print(f"  {emp_id}: {emp_data['name']} ({emp_data['department']})")

print("2. Активные сотрудники (active_names):")
active_names = service.active_names()
for name in active_names:
    print(f"  - {name}")

print("3. Сотрудники отдела 'Data' (department_members):")
data_members = service.department_members('Data')
for member in data_members:
    status = "активен" if member['is_active'] else "неактивен"
    print(f"  - {member['name']} ({member['email']}, {status})")

print("4. Email сотрудника с emp_id=2 (find_email):")
email = service.find_email(2)
print(f"  Email: {email}")

print("5. Группировка по отделам (group_by_department):")
grouped = service.group_by_department()
for department, names in grouped.items():
    print(f"  Отдел '{department}': {', '.join(names)}")
